# 01 · Explore the Hugging Face data

Goal: get a feeling for the data before building features.

WSL has only ~7 GB RAM. July 2024 (2 M rows) fits easily. For the big months from 2025-11 on (~15 M rows), select columns and filter **before** `.collect()`.

In [1]:
from pathlib import Path

import polars as pl

DATA_DIR = Path("../data/raw/hf/monthly_processed_data")
sorted(p.name for p in DATA_DIR.glob("*.parquet"))

['data-2024-07.parquet',
 'data-2024-08.parquet',
 'data-2024-09.parquet',
 'data-2024-10.parquet',
 'data-2024-11.parquet',
 'data-2024-12.parquet',
 'data-2025-01.parquet',
 'data-2025-02.parquet',
 'data-2025-03.parquet',
 'data-2025-04.parquet',
 'data-2025-05.parquet',
 'data-2025-06.parquet',
 'data-2025-07.parquet',
 'data-2025-08.parquet',
 'data-2025-09.parquet',
 'data-2025-10.parquet',
 'data-2025-11.parquet',
 'data-2025-12.parquet',
 'data-2026-01.parquet',
 'data-2026-02.parquet',
 'data-2026-03.parquet',
 'data-2026-04.parquet',
 'data-2026-05.parquet',
 'data-2026-06.parquet',
 'data-2026-07.parquet',
 'data-2026-08.parquet',
 'data-2026-09.parquet']

In [2]:
# scan = lazy: nothing is loaded yet, Polars only reads the file's schema
lf = pl.scan_parquet(DATA_DIR / "data-2024-07.parquet") # data-2026-06.parquet
lf.collect_schema()

Schema([('station_name', String),
        ('xml_station_name', String),
        ('eva', String),
        ('train_number', String),
        ('line_number', String),
        ('final_destination_station', String),
        ('delay_in_min', Int32),
        ('time', Datetime(time_unit='ns', time_zone=None)),
        ('arrival_is_canceled', Boolean),
        ('departure_is_canceled', Boolean),
        ('train_type', String),
        ('is_additional_stop', Boolean),
        ('is_replacement_train', Boolean),
        ('replaced_train_type', String),
        ('replaced_train_number', String),
        ('has_plan_record', Boolean),
        ('train_line_ride_id', String),
        ('train_line_station_num', Int32),
        ('arrival_planned_time', Datetime(time_unit='ns', time_zone=None)),
        ('arrival_change_time', Datetime(time_unit='ns', time_zone=None)),
        ('departure_planned_time', Datetime(time_unit='ns', time_zone=None)),
        ('departure_change_time', Datetime(time_unit='ns', t

In [3]:
df = lf.collect()  # now the data is loaded into memory
print(df.shape)
df.head()

(2029509, 23)


station_name,xml_station_name,eva,train_number,line_number,final_destination_station,delay_in_min,time,arrival_is_canceled,departure_is_canceled,train_type,is_additional_stop,is_replacement_train,replaced_train_type,replaced_train_number,has_plan_record,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id
str,str,str,str,str,str,i32,datetime[ns],bool,bool,str,bool,bool,str,str,bool,str,i32,datetime[ns],datetime[ns],datetime[ns],datetime[ns],str
null,"""ZOB/Hauptbahnhof, Pforzheim""","""0940370""","""33382""","""S6 (S""","""Bahnhof, Bad Wildbad""",0,2024-07-01 00:00:00,false,false,"""Bus""",false,false,null,null,true,"""-6129702905591104469""",1,null,null,2024-07-01 00:00:00,2024-07-01 00:00:00,"""-6129702905591104469-240701000…"
null,"""Hauptbahnhof, Saarbrücken""","""0836075""","""87312""","""1""","""Riegelsberg Süd, Riegelsberg""",0,2024-07-01 00:00:00,false,false,"""STB""",false,false,null,null,true,"""-788955727001224364""",9,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,"""-788955727001224364-2406302347…"
"""Bielefeld Hbf""","""Bielefeld Hbf""","""08000036""","""89739""","""6""","""Bielefeld Hbf""",-1,2024-07-01 00:00:00,false,false,"""RE""",false,false,null,null,true,"""2444314634072177884""",23,2024-07-01 00:01:00,2024-07-01 00:00:00,null,null,"""2444314634072177884-2406302048…"
"""Hamburg Hbf""","""Hamburg Hbf (S-Bahn)""","""08098549""","""45254""","""5""","""Hamburg Elbgaustraße""",0,2024-07-01 00:00:00,false,false,"""S""",false,false,null,null,true,"""1695252692593883835""",7,2024-06-30 23:59:00,2024-06-30 23:59:00,2024-07-01 00:00:00,2024-07-01 00:00:00,"""1695252692593883835-2406302341…"
"""Dresden Hbf""","""Dresden Hbf""","""08010085""","""31778""","""1""","""Meißen Triebischtal""",0,2024-07-01 00:00:00,false,false,"""S""",false,false,null,null,true,"""-2271920085331621501""",15,2024-06-30 23:58:00,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,"""-2271920085331621501-240630231…"


In [ ]:
# Arrival delay computed by ourselves (delay_in_min is mostly the departure)
arrivals = df.filter(
    pl.col("arrival_planned_time").is_not_null(),
    ~pl.col("arrival_is_canceled"),
).with_columns(
    arrival_delay_min=(pl.col("arrival_change_time") - pl.col("arrival_planned_time")).dt.total_minutes(),
).with_columns(
    y=(pl.col("arrival_delay_min") >= 6).cast(pl.Int8),
)
arrivals.select("station_name", "train_type", "train_number", "arrival_planned_time", "arrival_delay_min", "y").head(10)

In [ ]:
# Share of late arrivals per train type
arrivals.group_by("train_type").agg(
    pl.len().alias("stops"),
    pl.col("y").mean().round(3).alias("late_share"),
).sort("stops", descending=True).head(15)

## Questions to explore yourself

1. What does the distribution of `arrival_delay_min` look like? (Tip: `arrivals["arrival_delay_min"].describe()`, then a histogram)
2. At which hour of the day is the late share highest? (Tip: `pl.col("arrival_planned_time").dt.hour()`)
3. Which 10 stations have the highest late share?
4. How many stops have exactly 0 minutes delay? Is that plausible?
5. Pick one `train_line_ride_id` and look at all its stops sorted by `train_line_station_num`. How does the delay change along the ride?
6. Compare July 2024 with September 2026: did the late share change?